In [2]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jrobischon/wikipedia-movie-plots")

from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Hofstra/year3
%ls *.csv

#%cd /content/drive/MyDrive/TextMining/DataSets
#%ls *.csv


Using Colab cache for faster access to the 'wikipedia-movie-plots' dataset.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Hofstra/year3
'Copy of job_title_des.csv'   job_title_des.csv   wiki_movie_plots_deduped.csv


1. How many movies are in the dataset?
2. Select the movies after 1980's. How many movies do you have?
3. What are the 10 most frequent genres of movies?
4. Select the title, year, the movie plot, and the genre for the movies after 1980.
5. Combine into one column, the movie plot, the title, and the genre: "movie plot " + title + "." + genre + "."

In [3]:
movies_data = pd.read_csv('wiki_movie_plots_deduped.csv')

print(movies_data.head(1))

#testing if csv file can be read

   Release Year                   Title Origin/Ethnicity Director Cast  \
0          1901  Kansas Saloon Smashers         American  Unknown  NaN   

     Genre                                          Wiki Page  \
0  unknown  https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...   

                                                Plot  
0  A bartender is working at a saloon, serving dr...  


In [4]:
#Part 0
print(f"1. There are {movies_data.shape[0]} movies in the dataset\n")
#total movies in dataset, shape[0] returns number of rows

post1980 = movies_data[movies_data["Release Year"]> 1980]
print(f"2. There are {post1980.shape[0]} movies in the post 1980 dataset\n")
#returns movies with the value greater than 1980 for year column

topGenres = movies_data["Genre"].value_counts().head(10)
print(f"3. The 10 most frequent genres of movies are \n{topGenres} \n")
#value_counts returns total value for genre column, head(10) chooses only the top 10 at the top

specific_Data = post1980[["Title","Release Year", "Plot", "Genre"]]
#extracts only specific columns, didnt print out full table b/c too many
print(f"4. {specific_Data.head(3)} \n")

column = post1980["Combined"] = post1980["Plot"] + "." + post1980["Title"] + "." +  post1980["Genre"]
print(f"5. Top 3 from Combined Column \n{column.head(3)}")
#created new column that concats columns

1. There are 34886 movies in the dataset

2. There are 19994 movies in the post 1980 dataset

3. The 10 most frequent genres of movies are 
Genre
unknown      6083
drama        5964
comedy       4379
horror       1167
action       1098
thriller      966
romance       923
western       865
crime         568
adventure     526
Name: count, dtype: int64 

4.                    Title  Release Year  \
9796   Absence of Malice          1981   
9797      All Night Long          1981   
9798  ...All the Marbles          1981   

                                                   Plot          Genre  
9796  Miami liquor wholesaler Michael Gallagher (Pau...          drama  
9797  George Dupler (Gene Hackman), a married man ne...         comedy  
9798  Harry is the manager of a tag team of gorgeous...  comedy, drama   

5. Top 3 from Combined Column 
9796    Miami liquor wholesaler Michael Gallagher (Pau...
9797    George Dupler (Gene Hackman), a married man ne...
9798    Harry is the manager of a

/tmp/ipython-input-311326572.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  column = post1980["Combined"] = post1980["Plot"] + "." + post1980["Title"] + "." +  post1980["Genre"]


Part 1

Build the inverted index on the subset of movies (title + plot + genre) after 1980 (use the example we did in class (indexing_class.ipynb in the github repository). In addition, use spacy and the NER processing to extract names, locations, organizations - that span more than one word (e.g. San Francisco) - as terms to your vocabulary (look at the sample code in demo_spacy.ipynb).

1. Which nlp processing steps did you use?
2. What is the vocabulary size (# of unique terms after processing) after nlp processing?
3. Top 10 Most Freq
4. Top 10 Least Freq
5. Attempt to eliminate the most common and most rare terms. What is the size of your vocabulary now.
6. Give some examples of names, locations, or organizations you found and added to the corpus vocabulary.
7. How many unique such terms did you find?

In [5]:
#Part 1
from pprint import pp
from collections import defaultdict
import spacy

nlp = spacy.load('en_core_web_sm')

def compute_freq(lst):
    freq = defaultdict(int)
    for term in lst:
        freq[term] += 1
    return freq

def extract_terms(text):
    doc = nlp(text)
    out_terms = []

    #extract regular tokens lemmitized
    for token in doc:
        if token.is_alpha and not token.is_stop:
            out_terms.append(token.lemma_.lower())

    #extract multi-word named entities
    for ent in doc.ents:
        if ent.label_ in ['PERSON', 'LOC', 'ORG', 'GPE'] and len(ent.text.split()) > 1:
            entity_term = ent.text.lower().replace(' ', '_')
            out_terms.append(entity_term)

    return out_terms


def build_inverted_index(documents):
    inverted_index = defaultdict(dict)
    for doc_id, doc in enumerate(documents):
        terms_doc = extract_terms(doc)
        dict_doc = compute_freq(terms_doc)
        for term_doc, term_freq in dict_doc.items():
            inverted_index[term_doc][doc_id] = term_freq
    return inverted_index

def dist_query_docs(query_term_freq, inverted_index, topk=5):
    dot_product = defaultdict(int)
    for query_term, query_tf in query_term_freq.items():
        if query_term in inverted_index:
            dict_term_inverted_index = inverted_index[query_term]
            for doc_id, doc_tf in dict_term_inverted_index.items():
                dot_product[doc_id] += query_tf * doc_tf


    sorted_docs = sorted(dot_product.items(), key=lambda item: item[1], reverse=True)
    if len(sorted_docs) < topk:
        return sorted_docs
    else:
        return sorted_docs[:topk]

def print_docs(dict_doc_list, list_doc):
    for doc_id, dist_doc in dict_doc_list:
        print(f"doc_id dot product is {dist_doc}")
        pp(list_doc[doc_id])
        print()

def extract_entities(text):
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        if len(ent.text.split()) > 1:
            entities.append(ent.text.lower())
    return entities



In [6]:
#testing part 1 functions

print(f"Number of documents to process: {len(post1980)}")
print(f"Sample document length: {len(post1980['Combined'].iloc[0])}")
print(f"First document preview: {post1980['Combined'].iloc[0][:200]}...")

Number of documents to process: 19994
Sample document length: 3775
First document preview: Miami liquor wholesaler Michael Gallagher (Paul Newman), who is the son of a deceased criminal, awakes one day to find himself a front-page story in the local newspaper, indicating that he is being in...


In [7]:
import pickle
import os

#Part 1 Answers

# Check if we have saved data
if os.path.exists('inverted_index.pkl') and os.path.exists('documents.pkl'):
    print("Loading saved inverted index...")
    with open('inverted_index.pkl', 'rb') as f:
        inverted_index = defaultdict(dict, pickle.load(f))
    with open('documents.pkl', 'rb') as f:
        documents = pickle.load(f)
    print(f"Loaded {len(inverted_index)} terms and {len(documents)} documents\n")

else:

    documents = post1980["Combined"].tolist()
    print(f"Processing {len(documents)} documents")

    inverted_index = defaultdict(dict)

    inverted_index = defaultdict(dict)
    for doc_id, doc in enumerate(documents):
      terms = extract_terms(doc)
      term_freq = compute_freq(terms)
      for term, freq in term_freq.items():
          inverted_index[term][doc_id] = freq
    print("Building inverted index complete!")


    with open('inverted_index.pkl', 'wb') as f:
        pickle.dump(dict(inverted_index), f)
    with open('documents.pkl', 'wb') as f:
        pickle.dump(documents, f)
    print("Saved inverted index and documents!\n")

# Calculate document frequencies
doc_freq = {}
for term, doc_dict in inverted_index.items():
    doc_freq[term] = len(doc_dict)

# Sort by frequency
sorted_terms = sorted(doc_freq.items(), key=lambda x: x[1], reverse=True)

print(f"1. NLP steps: tokenization, lemmatization, stop word removal, Named Entity Recognition (NER)\n")
print(f"2. Vocabulary size: {len(inverted_index)} terms\n")
print(f"3. Most frequent 10 terms:")
for i, (term, freq) in enumerate(sorted_terms[:10]):
    print(f"   {term}: {freq} docs")
print(f"\n4. Least frequent 10 terms:")
for i, (term, freq) in enumerate(sorted_terms[-10:]):
    print(f"   {term}: {freq} docs")

# Filter vocabulary - remove very common and very rare terms
min_df = 2
max_df = int(len(documents) * 0.5)  # Changed from 0.8 to 0.5 (50%)
filtered_terms = [term for term, freq in doc_freq.items() if min_df <= freq <= max_df]
print(f"\n5. After filtering (min_df=2, max_df={max_df}): {len(filtered_terms)} terms left\n")

# Extract entity examples from first 1000 docs
print("Extracting entity examples...")
all_entities = []
for doc in documents[:1000]:  # Check first 1000 instead of 10
    doc_obj = nlp(doc)
    for ent in doc_obj.ents:
        if ent.label_ in ['PERSON', 'LOC', 'ORG', 'GPE'] and len(ent.text.split()) > 1:
            all_entities.append(ent.text.lower())

unique_entities = list(set(all_entities))
print(f"6. Entity examples: {unique_entities[:10]}")  # Show 10 examples
print(f"\n7. Total unique multi-word entities found (from first 1000 docs): {len(unique_entities)}")

Loading saved inverted index...
Loaded 179862 terms and 19994 documents

1. NLP steps: tokenization, lemmatization, stop word removal, Named Entity Recognition (NER)

2. Vocabulary size: 179862 terms

3. Most frequent 10 terms:
   find: 10802 docs
   leave: 9196 docs
   take: 8867 docs
   tell: 8202 docs
   friend: 8019 docs
   go: 7983 docs
   life: 7962 docs
   try: 7913 docs
   come: 7883 docs
   kill: 7787 docs

4. Least frequent 10 terms:
   axani: 1 docs
   reddit: 1 docs
   buzzfeed: 1 docs
   amy_tyler: 1 docs
   jordan_axani: 1 docs
   elizabeth_gallagher: 1 docs
   brendan_bradley: 1 docs
   soysal: 1 docs
   orhan_şahin: 1 docs
   deniz_soysal: 1 docs

5. After filtering (min_df=2, max_df=9997): 59808 terms left

Extracting entity examples...
6. Entity examples: ['merle webb', 'choke canyon.sci', 'the cycle lords', 'royal canadian mounted police', 'the sword of protection', 'david mendenhall', 'goth violet', 'anne osborne', 'robert morse', 'johnny steele']

7. Total unique m

In [8]:
from google.colab import files
files.download('inverted_index.pkl')
files.download('documents.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Part 2:

Come up with 3 queries related to movie plots. The queries should consist of more than 2-3 words. For example: 'Science fiction movies with aliens attacking earth'. Check if the most relevant words in your queries are in the inverted index - if they are not, then change your query, otherwise you will not find anything relevant. Process the queries using the same nlp processing steps as the documents. Hint: Do not use very common or very rare words in your queries.

For each query, display the query, its terms, and the frequency of each term

In [9]:
#Part 2
queries = ["scary movies to watch at night", "romantic comedy movies to watch for fun", "worst action movies of all time"]

for i, query in enumerate(queries):
  print(f"\nQuery {i+1}: {query}")
  query_terms = extract_terms(query)
  query_freq = compute_freq(query_terms)

  print(f"Terms: {query_terms}")
  print(f"Term frequencies: {query_freq}")

  #search for terms in index
  print("Terms in vocabulary:")
  for term in query_terms:
      if term in inverted_index:
          doc_count = len(inverted_index[term])
          print(f"  {term}: found in {doc_count} docs")
      else:
          print(f"  {term}: NOT found")


Query 1: scary movies to watch at night
Terms: ['scary', 'movie', 'watch', 'night']
Term frequencies: defaultdict(<class 'int'>, {'scary': 1, 'movie': 1, 'watch': 1, 'night': 1})
Terms in vocabulary:
  scary: found in 34 docs
  movie: found in 2435 docs
  watch: found in 2398 docs
  night: found in 4779 docs

Query 2: romantic comedy movies to watch for fun
Terms: ['romantic', 'comedy', 'movie', 'watch', 'fun']
Term frequencies: defaultdict(<class 'int'>, {'romantic': 1, 'comedy': 1, 'movie': 1, 'watch': 1, 'fun': 1})
Terms in vocabulary:
  romantic: found in 759 docs
  comedy: found in 1234 docs
  movie: found in 2435 docs
  watch: found in 2398 docs
  fun: found in 476 docs

Query 3: worst action movies of all time
Terms: ['bad', 'action', 'movie', 'time']
Term frequencies: defaultdict(<class 'int'>, {'bad': 1, 'action': 1, 'movie': 1, 'time': 1})
Terms in vocabulary:
  bad: found in 1588 docs
  action: found in 1656 docs
  movie: found in 2435 docs
  time: found in 7732 docs


3. Write a function to compute the similarity of a query with the collection. Do this efficiently by computing the similarity between a query and the set of documents that contain at least two words from the query. Implement two different TF-IDF similarity functions (one of them should be BM25 - choose another one from the slides or textbook). Retrieve the ranked list of the first 7 most similar movies for each of your queries and for each of the two similarity functions.

  For each similarity function, and each query, display the closest 7 titles, plots and genres (not the full text) and their similarity values.


In [10]:
import math

# Calculate IDF
num_docs = len(documents)
idf = {}
for term in filtered_terms:
    if term in inverted_index:
        idf[term] = math.log((num_docs + 1) / (len(inverted_index[term]) + 1)) + 1

# Pre-calculate document lengths ONCE
print("Calculating document lengths")
doc_lengths = [len(extract_terms(doc)) for doc in documents]
avg_doc_length = sum(doc_lengths) / len(doc_lengths)


def compute_sim_query_tfidf(query_terms, inverted_index, idf, documents, topk=7):
    query_term_freq = compute_freq(query_terms)
    document_scores = defaultdict(float)
    for query_term, query_tf in query_term_freq.items():
        if query_term in idf and query_term in inverted_index:
            query_tfidf = query_tf * idf[query_term]
            for doc_id, doc_tf in inverted_index[query_term].items():
                doc_tfidf = doc_tf * idf[query_term]
                document_scores[doc_id] += query_tfidf * doc_tfidf
    sorted_docs = sorted(document_scores.items(), key=lambda item: item[1], reverse=True)
    return sorted_docs[:topk]

def compute_sim_query_bm25(query_terms, inverted_index, idf, topk=7, k1=1.5, b=0.75):
    query_term_freq = compute_freq(query_terms)
    document_scores = defaultdict(float)
    for query_term, query_tf in query_term_freq.items():
        if query_term in idf and query_term in inverted_index:
            idf_score = idf[query_term]
            for doc_id, doc_tf in inverted_index[query_term].items():
                numerator = doc_tf * (k1 + 1)
                denominator = doc_tf + k1 * (1 - b + b * (doc_lengths[doc_id] / avg_doc_length))
                term_score = idf_score * (numerator / denominator)
                document_scores[doc_id] += term_score
    sorted_docs = sorted(document_scores.items(), key=lambda item: item[1], reverse=True)
    return sorted_docs[:topk]

Calculating document lengths


In [11]:
# Calculate IDF for terms in the filtered vocabulary
num_docs = len(documents)
idf = {}
for term in filtered_terms:
    if term in inverted_index:
        # Add 1 smoothing to avoid division by zero
        idf[term] = math.log((num_docs + 1) / (len(inverted_index[term]) + 1)) + 1

print("IDF calculated.")
print("\nSample IDF values:")
# Print a sample of IDF values
for i, (term, value) in enumerate(idf.items()):
    if i < 10:
        print(f"  {term}: {value:.4f}")
    else:
        break

IDF calculated.

Sample IDF values:
  miami: 6.1758
  liquor: 5.9906
  wholesaler: 9.5169
  michael: 4.1405
  gallagher: 7.4693
  paul: 4.5113
  newman: 7.4693
  son: 2.3882
  deceased: 4.9657
  criminal: 3.7533


In [12]:
def compute_sim_query_tfidf(query_terms, inverted_index, idf, documents, topk=7):
    query_term_freq = compute_freq(query_terms)
    document_scores = defaultdict(float)

    for query_term, query_tf in query_term_freq.items():
        if query_term in idf and query_term in inverted_index:
            query_tfidf = query_tf * idf[query_term]

            for doc_id, doc_tf in inverted_index[query_term].items():
                doc_tfidf = doc_tf * idf[query_term]
                document_scores[doc_id] += query_tfidf * doc_tfidf

    sorted_docs = sorted(document_scores.items(), key=lambda item: item[1], reverse=True)
    return sorted_docs[:topk]

In [13]:
#  query based from the list defined
query1 = queries[0]
query1_terms = extract_terms(query1)

# Compute similarity using TF-IDF
tfidf_results = compute_sim_query_tfidf(query1_terms, inverted_index, idf, documents)

# Helper function to print a few lines
def print_few_lines(text, num_lines=2):
    lines = text.splitlines()
    for i in range(min(num_lines, len(lines))):
        print(lines[i])
    print("...")

#tests to see if dot product calculate along witht the similar terms
# print(f"TF-IDF similarity results for query: '{query1}'")
# for doc_id, dist_doc in tfidf_results:
#     print(f"doc_id dot product is {dist_doc}")
#     print_few_lines(documents[doc_id])
#     print()

In [16]:
#part 3 results

for query in queries:
    print(f"\n\nQuery: {query}")

    query_terms = extract_terms(query)

    # TF-IDF Results
    print("\nTF-IDF Results:")
    tfidf_results = compute_sim_query_tfidf(query_terms, inverted_index, idf, documents)

    for doc_id, score in tfidf_results:
        original_index = post1980.iloc[doc_id].name
        print(f"\nScore: {score:.2f}")
        print(f"Title: {post1980.loc[original_index, 'Title']}")
        print(f"Genre: {post1980.loc[original_index, 'Genre']}")
        print(f"Plot: {post1980.loc[original_index, 'Plot'][:200]}...")

    # BM25 Results
    print("\n\nBM25 Results:")
    bm25_results = compute_sim_query_bm25(query_terms, inverted_index, idf)

    for doc_id, score in bm25_results:
        original_index = post1980.iloc[doc_id].name
        print(f"\nScore: {score:.2f}")
        print(f"Title: {post1980.loc[original_index, 'Title']}")
        print(f"Genre: {post1980.loc[original_index, 'Genre']}")
        print(f"Plot: {post1980.loc[original_index, 'Plot'][:200]}...")



Query: scary movies to watch at night

TF-IDF Results:

Score: 357.33
Title: Dark Tales of Japan
Genre: horror
Plot: Introduction: Would You Like to Hear a Scary Tale? (Intorodakushon: Kowai hanashi, kikitai desu ka) Directed by Yoshihiro Nakamura; teleplay by Yoshihiro Nakamura and Katsuhide Suzuki
Plot: At a bus ...

Score: 235.25
Title: Winter
Genre: horror
Plot: Jayaram and Bhavana takes the lead roles as Dr. Ramdas and his wife in the film. Dr. Ramdas is leading medical practitioner in Hyderabad. Fed up with the fast-paced city life, Ramdas and his wife deci...

Score: 182.21
Title: Movie 43
Genre: comedy
Plot: Movie 43 is a series of different sketches containing different scenes and scenarios.
The film is composed of multiple comedy shorts presented through an overarching segment titled "The Pitch", in wh...

Score: 172.03
Title: Darna Zaroori Hai
Genre: horror
Plot: Darna Zaroori Hai interweaves six stories into one film. Five children get lost in the middle of a forest until

**Part 4: Top 7 Precision Calculation**

For each query and each similarity function, you will manually assess the top 7 retrieved documents for relevance and calculate the precision.

In [15]:
#part 4 percison
def calculate_precision(results, documents, query, method_name, post1980_df):
    """
    Calculates the top 7 precision based on manual relevance judgments.

    Args:
        results (list): List of tuples containing (document_id, similarity_score) for top 7 results.
        documents (list): List of document texts.
        query (str): The original query string.
        method_name (str): The name of the similarity method (e.g., "TF-IDF", "BM25").
        post1980_df (pd.DataFrame): The original dataframe with movie information.


    Returns:
        float: The top 7 precision value.
    """
    print(f"\nAssessing relevance for query: '{query}' using {method_name}")
    relevant_count = 0
    for i, (doc_id, score) in enumerate(results):
        # Retrieve the original movie information using the doc_id
        original_index = post1980_df.iloc[doc_id].name
        movie_title = post1980_df.loc[original_index, "Title"]
        movie_plot = post1980_df.loc[original_index, "Plot"]
        movie_genre = post1980_df.loc[original_index, "Genre"]

        print(f"\nResult {i+1} (Doc ID: {doc_id}, Score: {score:.4f})")
        print(f"Title: {movie_title}")
        print(f"Genre: {movie_genre}")
        print(f"Plot: {movie_plot[:200]}...") # Print only first 200 characters of plot


        #  MANUALLY JUDGE RELEVANCE HERE
        # Replace the input() with your judgment ( is_relevant = True or False)
        is_relevant = input("Is this document relevant? (yes/no): ").lower() == 'yes'


        if is_relevant:
            relevant_count += 1

    precision = relevant_count / len(results)
    print(f"\nTop 7 Precision for query '{query}' ({method_name}): {precision:.4f}")
    return precision

# Iterate through each query
for i, query in enumerate(queries):
    print(f"\n--- Processing Query {i+1}: '{query}' ---")
    query_terms = extract_terms(query)

    # Compute similarity using TF-IDF
    tfidf_results = compute_sim_query_tfidf(query_terms, inverted_index, idf, documents)

    # Calculate and display precision for TF-IDF
    precision_tfidf = calculate_precision(tfidf_results, documents, query, "TF-IDF", post1980)

    # Compute similarity using BM25
    bm25_results = compute_sim_query_bm25(query_terms, inverted_index, idf)

    # Calculate and display precision for BM25
    precision_bm25 = calculate_precision(bm25_results, documents, query, "BM25", post1980)


--- Processing Query 1: 'scary movies to watch at night' ---

Assessing relevance for query: 'scary movies to watch at night' using TF-IDF

Result 1 (Doc ID: 18567, Score: 357.3258)
Title: Dark Tales of Japan
Genre: horror
Plot: Introduction: Would You Like to Hear a Scary Tale? (Intorodakushon: Kowai hanashi, kikitai desu ka) Directed by Yoshihiro Nakamura; teleplay by Yoshihiro Nakamura and Katsuhide Suzuki
Plot: At a bus ...
Is this document relevant? (yes/no): yes

Result 2 (Doc ID: 14563, Score: 235.2495)
Title: Winter
Genre: horror
Plot: Jayaram and Bhavana takes the lead roles as Dr. Ramdas and his wife in the film. Dr. Ramdas is leading medical practitioner in Hyderabad. Fed up with the fast-paced city life, Ramdas and his wife deci...
Is this document relevant? (yes/no): yes

Result 3 (Doc ID: 6820, Score: 182.2122)
Title: Movie 43
Genre: comedy
Plot: Movie 43 is a series of different sketches containing different scenes and scenarios.
The film is composed of multiple comedy 

**Comparing TF-IDF and BM25 Precision Results**

After running the code cell and manually judging the relevance of the top 7 results for each query and similarity function, you will have the top 7 precision score for each combination. Use these scores to compare the performance of the standard TF-IDF and BM25 methods.


1.  State the Precision Values:

  Query 1 ("scary movies to watch at night"): TF-IDF Precision = 0.4286, BM25 Precision = 0.4286

  Query 2 ("romantic comedy movies to watch for fun"): TF-IDF Precision = 0.0000, BM25 Precision = 0.7143

  Query 3 ("worst action movies of all time"): TF-IDF Precision = 0.1429, BM25 Precision = 0.4286

2.  Direct Comparison:

  Query 1, both methods performed equally, this leads us to believe that when the dataset matches the horror theme, both methods retunr similar results. Query 2 however resulted in a major difference where TF-IDF failed to retrieve relevant romantic comedies while BM25 resulted in a much higher score of 0.7143. For Query 3, BM25 once again outperformed TF-IDF, resulting more relevant action films with a precision of .0.4286 while TF-IDF resulted in 0.1429.

3.   Reasons for Differences/conclusion:

These results demonstrate that although TF-IDF and BMD25 can be compared when query is simple, BM25 overall outperforms retrieval when relevance depends on more refined rankings. BM25's edge in Query 2 depicts its higher strength in retrieval, overweighting popular terms so that i can rank highly where the "romantic" element counts as much as teh "comedy" one.

Overall, BM25 slightly outperformed TF-IDF in this experiment, particularly for queries requiring more refined ranking of relevance. The differences can be attributed to BM25’s ability to balance term frequency without overweighting repeated terms, making it better suited for capturing meaningful results in queries with less direct matches. In contrast, TF-IDF may struggle with cases where raw term frequency skews the relevance ranking.
